# 01 -- Data loading

Convert the Arctic Shift archives of r/litigi to Parquet files in the schema of [subreddit-lens](https://pypi.org/project/subreddit-lens/). Settings come from `subreddit-lens.toml` next to this notebook.

- Input: `data/raw/litigi_comments.zst` (required) and `data/raw/litigi_submissions.zst` (optional: needed to count replies to the author of each post), shared with the training pipeline and only read here.
- Output: `data/analytics/litigi_comments.parquet` and `litigi_submissions.parquet`. The training pipeline's own Parquet files in `data/raw/` have a different schema and are not touched.

The archives are streamed and written in chunks, so memory use does not depend on the archive size. If an archive is missing but its Parquet file already exists, the existing Parquet file is kept.

In [ ]:
import logging
from pathlib import Path

from subreddit_lens import ingest_archive, load_comments, load_config

logging.basicConfig(level=logging.INFO, format="%(message)s")

# Locate the analytics directory, which holds subreddit-lens.toml, so paths
# work regardless of the working directory.
ANALYTICS_DIR = next(
    p
    for p in [Path.cwd(), Path.cwd() / "analytics", *Path.cwd().parents]
    if (p / "subreddit-lens.toml").exists()
)
config = load_config(ANALYTICS_DIR / "subreddit-lens.toml")
config

## Comments

In [ ]:
if config.comments_archive.exists():
    n_comments = ingest_archive(
        config.comments_archive,
        config.comments_parquet,
        kind="comments",
        condition=config.date_filter(),
    )
    print(f"{n_comments:,} comments written to {config.comments_parquet}")
elif config.comments_parquet.exists():
    print(f"{config.comments_archive} not found: keeping {config.comments_parquet}.")
else:
    raise FileNotFoundError(
        f"Neither {config.comments_archive} nor {config.comments_parquet} exists: "
        "download the r/litigi comments archive from Arctic Shift first."
    )

## Submissions (optional)

In [ ]:
if config.submissions_archive.exists():
    n_submissions = ingest_archive(
        config.submissions_archive,
        config.submissions_parquet,
        kind="submissions",
        condition=config.date_filter(),
    )
else:
    print(f"{config.submissions_archive} not found: skipping submissions.")

## Check

In [ ]:
comments = load_comments(config.comments_parquet)
print(f"{len(comments):,} comments from {comments['author'].nunique():,} authors")
print(f"{comments['created_dt'].min()} -> {comments['created_dt'].max()}")
comments.head()